# Phase 4: Causal Patching — Activation Patching

**Experimental protocol:**
1. Run **counterfactual prompt** → save hidden state at Layer 10 (last token)
2. Run **clean prompt** normally → save the baseline answer
3. Run **clean prompt again**, but REPLACE Layer 10's hidden state (last token of prefill) with the CF state harvested in step 1
4. Measure whether the answer changed

If the answer changes in ≥50% of cases, Layer 10 is causally responsible for the hop-1 reasoning step.

In [ ]:
!pip install -q transformers accelerate huggingface_hub
import huggingface_hub
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("Logged in via Kaggle Secrets!")
except Exception as e:
    print(f"Kaggle Secrets failed ({e}), using fallback...")
    hf_token = "REPLACE_ME_WITH_YOUR_RAW_HF_TOKEN"
    huggingface_hub.login(token=hf_token)

In [ ]:
import json, torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
TARGET_LAYER = 10

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print(f"Model loaded. Device: {model.device}")

In [ ]:
print("Downloading dataset...")
file_path = hf_hub_download(
    repo_id="AnishRacherla/LinguaFranca-Phase3",
    filename="data/2wikimultihopqa/augmented.jsonl",
    repo_type="dataset"
)
with open(file_path) as f:
    data = [json.loads(line) for line in f]

counterfactuals = [d for d in data if d.get("is_counterfactual", False)]
cleans = {d["id"]: d for d in data if not d.get("is_counterfactual", False)}
print(f"Loaded {len(cleans)} clean examples, {len(counterfactuals)} counterfactuals.")

# Build minimal pairs: clean example answered correctly (hop1 label=0) 
# AND its CF twin answered incorrectly (hop1 label=1)
pairs = []
for cf in counterfactuals:
    clean = cleans.get(cf.get("clean_pair_id"))
    if not clean:
        continue
    clean_h1 = next((h for h in clean.get("hops", []) if h["hop_idx"] == 1), None)
    cf_h1 = next((h for h in cf.get("hops", []) if h["hop_idx"] == 1), None)
    if clean_h1 and cf_h1 and clean_h1.get("label") == 0 and cf_h1.get("label") == 1:
        pairs.append((clean, cf))
            

print(f"Found {len(pairs)} perfect minimal pairs.")

In [ ]:
def generate_answer(prompt, max_new_tokens=60, hook_fn=None, layer_idx=None):
    """Generate an answer for a prompt. Optionally attach a hook to a layer."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    handle = None
    if hook_fn is not None and layer_idx is not None:
        handle = model.model.layers[layer_idx].register_forward_hook(hook_fn)
    
    try:
        with torch.no_grad():
            out_ids = model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
    finally:
        if handle:
            handle.remove()
    
    prompt_len = inputs.input_ids.shape[1]
    gen_ids = out_ids[0][prompt_len:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


def extract_answer_tag(text):
    """Pull out just the <answer>...</answer> part."""
    import re
    m = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if m:
        return m.group(1).strip().lower()
    # Fallback: last non-empty line
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    return lines[-1].lower() if lines else ""


print(f"Running causal patching on {len(pairs)} pairs at Layer {TARGET_LAYER}...")

n_changed = 0       # Answer changed after injection (successful corruption)
n_already_wrong = 0 # Baseline was already wrong (shouldn't happen with our pairs)
n_total = 0
examples_log = []

for i, (clean, cf) in enumerate(tqdm(pairs, desc="Causal Patching")):
    cf_prompt = cf["prompt"]
    clean_prompt = clean["prompt"]
    
    # Step 1: Get the CF hidden state at layer TARGET_LAYER (last token of prefill)
    cf_state = {}
    def harvest_hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        cf_state['vec'] = hidden[0, -1, :].clone().detach()
        return output
    
    cf_inputs = tokenizer(cf_prompt, return_tensors="pt").to(model.device)
    harvest_handle = model.model.layers[TARGET_LAYER].register_forward_hook(harvest_hook)
    with torch.no_grad():
        model(**cf_inputs)  # just a forward pass, no generation needed
    harvest_handle.remove()
    cf_vec = cf_state['vec']
    
    # Step 2: Get the BASELINE answer for the clean prompt (no injection)
    baseline_text = generate_answer(clean_prompt)
    baseline_ans = extract_answer_tag(baseline_text)
    
    # 2b. Check if baseline is ALREADY wrong (due to greedy decode float16 drift)
    expected_ans = clean.get("predicted_answer", clean.get("answer", ""))
    if expected_ans and expected_ans.lower() not in baseline_ans:
        n_already_wrong += 1
        continue
    
    # Only increment denominator if baseline was correct
    n_total += 1
    
    # Step 3: Generate WITH the CF hidden state injected into the clean prompt
    injected = [False]
    def inject_hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        rest = output[1:] if isinstance(output, tuple) else None
        # Only inject ONCE during the prefill (seq_len > 1)
        if not injected[0] and hidden.shape[1] > 1:
            hidden = hidden.clone()
            hidden[0, -1, :] = cf_vec
            injected[0] = True
        if rest is not None:
            return (hidden,) + rest
        return hidden
    
    corrupted_text = generate_answer(clean_prompt, hook_fn=inject_hook, layer_idx=TARGET_LAYER)
    corrupted_ans = extract_answer_tag(corrupted_text)
    
    # Did the answer change?
    changed = (baseline_ans != corrupted_ans)
    if changed:
        n_changed += 1
    
    examples_log.append({
        "clean_q": clean["question"],
        "cf_q": cf["question"],
        "baseline_ans": baseline_ans,
        "corrupted_ans": corrupted_ans,
        "changed": changed
    })
    
    if i < 3:  # Print first 3 examples
        print(f"\n--- Example {i+1} ---")
        print(f"Clean Q:       {clean['question']}")
        print(f"CF Q:          {cf['question']}")
        print(f"Baseline ans:  {baseline_ans}")
        print(f"Corrupted ans: {corrupted_ans}")
        print(f"Changed?       {'✅ YES' if changed else '❌ NO'}")

rate = n_changed / n_total * 100 if n_total > 0 else 0
print(f"\n{'='*40}")
print(f"=== CAUSAL PATCHING RESULTS ===")
print(f"{'='*40}")
print(f"Total pairs tested:       {n_total}")
print(f"Answers changed:          {n_changed}")
print(f"Causal Corruption Rate:   {rate:.1f}%")
print(f"")
print(f"Interpretation:")
if rate >= 50:
    print(f"  ✅ STRONG CAUSAL EFFECT: Layer {TARGET_LAYER} is causally responsible for")
    print(f"     hop-1 reasoning. Injecting CF state corrupts {rate:.0f}% of clean answers.")
elif rate >= 20:
    print(f"  ⚠️  MODERATE CAUSAL EFFECT: Layer {TARGET_LAYER} has partial influence.")
    print(f"     Consider patching a range of layers (e.g. 8-12) for stronger effect.")
else:
    print(f"  ❌ WEAK CAUSAL EFFECT: Layer {TARGET_LAYER} may not be the right bottleneck.")
    print(f"     Consider patching multiple layers or using mean-pooled states.")